# SHBT.205 Speech Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/alpha/tutorials/audio/shbt205_lab.ipynb)

This lab notebook walks through core speech analysis tasks using **senselab**:

1. **Recording or loading** an audio sample
2. **Preprocessing** -- resampling, visualization
3. **Automatic Speech Recognition (ASR)** with Whisper via senselab
4. **Articulatory coding** with SPARC (Speech Articulatory Coding)
5. **Phonetic Posteriorgram (PPG)** analysis and phoneme timing

Each section replaces raw library calls with senselab's unified API, giving
you reproducible, device-aware pipelines with minimal boilerplate.

## Setup

Install senselab and its dependencies. This single install provides
Whisper ASR, SPARC articulatory coding, and PPG extraction -- all
managed through senselab's unified interface.

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab"

> **\u26a0\ufe0f Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime \u2192 Restart session**, then **run all cells below** (skip this install cell).


## Imports and device selection

We import the senselab modules we need and auto-detect whether a GPU is
available. senselab uses `DeviceType` to abstract device selection across
platforms.

In [ ]:
import os

import torch

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.preprocessing import downmix_audios_to_mono, resample_audios
from senselab.audio.tasks.speech_to_text import transcribe_audios
from senselab.utils.data_structures import DeviceType, HFModel

%matplotlib inline

# Auto-detect device (CUDA > CPU; MPS not yet supported by all backends)
device = DeviceType.CUDA if torch.cuda.is_available() else DeviceType.CPU
print(f"Using device: {device.value}")

## Record or load audio

In a live Colab session you can record audio directly from your
microphone using the `ipywebrtc` widget. When running non-interactively
(e.g. in CI or as a script), we fall back to downloading a sample audio
file from the senselab test data.

In [ ]:
import urllib.request
from pathlib import Path

AUDIO_DIR = Path("tutorial_audio_files")
AUDIO_DIR.mkdir(exist_ok=True)

SAMPLE_URL = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing/audio_48khz_mono_16bits.wav"

filename = str(AUDIO_DIR / "recorded_audio.wav")

recording_widget_available = False
try:
    from ipywebrtc import AudioRecorder, CameraStream

    camera = CameraStream(constraints={"audio": True, "video": False})
    recorder = AudioRecorder(stream=camera)
    recording_widget_available = True
    print("Recording widget loaded. Press the circle to record, then press again to stop.")
    display(recorder)  # noqa: F821
except Exception:
    print("Recording widget not available -- downloading sample audio.")
    urllib.request.urlretrieve(SAMPLE_URL, filename)
    print(f"Saved sample audio to {filename}")

In [ ]:
# If you used the recording widget, run this cell to save your recording.
# If you used the fallback sample, this cell is a no-op.
if recording_widget_available:
    with open("recording.webm", "wb") as f:
        f.write(recorder.audio.value)
    os.system(f"ffmpeg -i recording.webm -ac 1 -f wav {filename} -y -hide_banner -loglevel panic")
    print(f"Saved recording to {filename}")
else:
    print(f"Using sample audio: {filename}")

## Load audio into senselab

The `Audio` class is senselab's core data container. It lazily loads the
waveform and stores metadata (sampling rate, number of channels, etc.).

In [ ]:
audio = Audio(filepath=os.path.abspath(filename))
print(f"Sampling rate : {audio.sampling_rate} Hz")
print(f"Channels      : {audio.waveform.shape[0]}")
print(f"Samples       : {audio.waveform.shape[1]}")
print(f"Duration      : {audio.waveform.shape[1] / audio.sampling_rate:.2f} s")

---

# Audio Preprocessing

Most speech models expect mono audio at a specific sampling rate (often
16 kHz). senselab provides `resample_audios` and `downmix_audios_to_mono`
as batch-friendly preprocessing utilities.

## Resampling to 16 kHz

In [ ]:
# Ensure mono
if audio.waveform.shape[0] > 1:
    audio = downmix_audios_to_mono([audio])[0]

print(f"Original sampling rate: {audio.sampling_rate} Hz")
[audio16k] = resample_audios([audio], resample_rate=16000)
print(f"Resampled sampling rate: {audio16k.sampling_rate} Hz")

## Visualize waveforms and spectrograms

senselab includes plotting utilities that work directly with `Audio` objects.

In [ ]:
from senselab.audio.tasks.plotting.plotting import play_audio, plot_specgram, plot_waveform

print("--- Original audio ---")
play_audio(audio)
plot_waveform(audio)

print("\n--- Resampled to 16 kHz ---")
play_audio(audio16k)
plot_waveform(audio16k)

In [ ]:
print("--- Original spectrogram ---")
plot_specgram(audio)

print("\n--- 16 kHz spectrogram ---")
plot_specgram(audio16k)

---

# Speech-to-Text with Whisper

OpenAI's [Whisper](https://openai.com/blog/whisper/) is a general-purpose
ASR model trained on 680k hours of multilingual data. senselab wraps the
Hugging Face Transformers pipeline so you get a clean, reproducible API
with device management.

**Key difference from raw Whisper**: instead of loading the model
manually and managing mel-spectrograms yourself, senselab's
`transcribe_audios` handles everything -- you just pass `Audio` objects
and a model specification.

In [ ]:
# Specify the Whisper model via HuggingFace model ID
asr_model = HFModel(path_or_uri="openai/whisper-base", revision="main")

# Transcribe -- senselab handles mel-spectrogram creation internally.
# The audio should be mono 16 kHz for best results.
transcripts = transcribe_audios(
    audios=[audio16k],
    model=asr_model,
    device=device,
)

print("Transcription:")
print(transcripts[0].text)

### Lab question: How well did Whisper do?

Consider what aspects of the audio recording could impact the
transcription accuracy:
- Background noise
- Microphone quality
- Speaking rate and clarity
- Accent and language

---

# Speech Articulatory Coding (SPARC)

[SPARC](https://arxiv.org/abs/2406.12998) is a neural encoding-decoding
framework that represents speech as **14 articulatory features** at 50 Hz,
plus a disentangled speaker embedding. Each channel corresponds to a
physical articulator on the vocal tract:

| Abbreviation | Articulator |
|---|---|
| TD (x, y) | Tongue Dorsum |
| TB (x, y) | Tongue Body |
| TT (x, y) | Tongue Tip |
| LI (x, y) | Lower Incisor (jaw) |
| UL (x, y) | Upper Lip |
| LL (x, y) | Lower Lip |

senselab wraps SPARC through `SparcFeatureExtractor`, which runs in an
isolated subprocess environment -- no extra installation needed.

In [ ]:
from senselab.audio.tasks.features_extraction.sparc import SparcFeatureExtractor

# SPARC expects mono audio. We use audio16k (already mono).
# Set resample=True so senselab resamples to the model's expected rate if needed.
sparc_features = SparcFeatureExtractor.extract_sparc_features(
    audios=[audio16k],
    device=device,
    resample=True,
)

features = sparc_features[0]
print("SPARC feature keys and shapes:")
for key, val in features.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.shape}")
    else:
        print(f"  {key}: {val}")

## Visualize articulatory trajectories

The EMA (electromagnetic articulography) features have 12 channels
corresponding to x/y coordinates of 6 articulators. We plot them as
time series, vertically offset for clarity.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Color scheme for articulators
COLOR_CODE = {
    "UL": matplotlib.colors.to_rgb("#EE3A5B"),
    "LL": matplotlib.colors.to_rgb("#FFD155"),
    "LI": matplotlib.colors.to_rgb("#959595"),
    "TT": matplotlib.colors.to_rgb("#43B962"),
    "TB": matplotlib.colors.to_rgb("#C44B9F"),
    "TD": matplotlib.colors.to_rgb("#0093B7"),
}

EMA_LABELS = ["TDX", "TDY", "TBX", "TBY", "TTX", "TTY", "LIX", "LIY", "ULX", "ULY", "LLX", "LLY"]
DISPLAY_ORDER = ["UL", "LL", "LI", "TT", "TB", "TD"]


def plot_articulatory_trajectories(ema_tensor: torch.Tensor, max_frames: int = 400, gap: int = 6) -> None:
    """Plot EMA articulatory trajectories."""
    ema = ema_tensor.numpy() if isinstance(ema_tensor, torch.Tensor) else ema_tensor
    ema = ema[:max_frames]

    # Build channel indices in display order
    ch_indices = []
    for label in DISPLAY_ORDER:
        ch_indices.append(EMA_LABELS.index(label + "X"))
        ch_indices.append(EMA_LABELS.index(label + "Y"))

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    yticks, ytick_labels = [], []

    for i, ch_i in enumerate(ch_indices):
        ch_label = EMA_LABELS[ch_i]
        art_name = ch_label[:2]
        color = COLOR_CODE[art_name]
        ax.plot(ema[:, ch_i] - gap * i, color=color, lw=2)
        yticks.append(-gap * i)
        ytick_labels.append(ch_label)

    ax.set_yticks(yticks)
    ax.set_yticklabels(ytick_labels, fontsize=12)

    # Time axis in seconds (50 Hz frame rate)
    xticks = np.arange(0, len(ema), 50)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{int(x * 20 / 1000)}" for x in xticks], fontsize=12)
    ax.set_xlabel("Time (s)", fontsize=13)
    ax.set_xlim(0, len(ema))
    ax.set_title("SPARC Articulatory Trajectories (EMA)", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_articulatory_trajectories(features["ema"])

### Interpreting the plot

Each pair of lines (X and Y) shows the estimated horizontal and vertical
position of an articulator over time. You can observe:
- **Tongue tip (TT)** movements during consonants
- **Jaw (LI)** opening/closing for vowels
- **Lip (UL, LL)** rounding and spreading

The SPARC model also extracts **pitch**, **loudness**, **periodicity**,
and a **speaker embedding** -- all accessible from the features dict.

In [ ]:
# Plot pitch and loudness from SPARC
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

pitch = features["pitch"].numpy().squeeze()
loudness = features["loudness"].numpy().squeeze()

# Each feature may have a slightly different frame count; build separate time axes
pitch_time = np.arange(len(pitch)) * 0.02  # 50 Hz -> 20 ms per frame
loudness_time = np.arange(len(loudness)) * 0.02

axes[0].plot(pitch_time, pitch, color="#FB754D", lw=1.5)
axes[0].set_ylabel("Pitch (Hz)")
axes[0].set_title("SPARC Pitch and Loudness")

axes[1].plot(loudness_time, loudness, color="#0093B7", lw=1.5)
axes[1].set_ylabel("Loudness")
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

---

# Phonetic Posteriorgrams (PPGs)

[PPGs](https://www.maxrmorrison.com/sites/ppgs/) provide a different
representation of speech -- instead of articulatory (physical) space,
they map into **phonemic (symbolic) space**. For each time frame, a PPG
gives the posterior probability of each phoneme being spoken.

This is based on the [ppgs library](https://github.com/interactiveaudiolab/ppgs)
and the paper by [Morrison et al. (2024)](https://www.maxrmorrison.com/pdfs/morrison2024fine.pdf).

senselab wraps PPG extraction through `extract_ppgs_from_audios`, which
also runs in an isolated subprocess environment.

In [ ]:
from senselab.audio.tasks.features_extraction.ppg import (
    extract_mean_phoneme_durations,
    extract_ppgs_from_audios,
    plot_ppg_phoneme_timeline,
)

# Extract PPGs -- audio must be mono
ppgs = extract_ppgs_from_audios(audios=[audio16k], device=device)
ppg = ppgs[0]

print(f"PPG shape: {ppg.shape}")
print(f"  (1 x {ppg.shape[1]} phonemes x {ppg.shape[2]} frames)")

## PPG phoneme timeline

The timeline shows when each phoneme is dominant (has the highest
posterior probability). This gives you a visual "transcription" at
the phoneme level.

In [ ]:
fig = plot_ppg_phoneme_timeline(audio16k, ppg, title="PPG Phoneme Timeline", show=True)

## Phoneme duration statistics

senselab can also compute per-phoneme duration statistics from the PPG.
This is useful for studying speaking rate, rhythm, and articulation
patterns.

In [ ]:
duration_stats = extract_mean_phoneme_durations(audio16k, ppg)

print(f"Total frames        : {duration_stats.get('frame_count', 'N/A')}")
print(f"Analysis duration   : {duration_stats.get('analysis_duration_seconds', 0):.2f} s")
print(f"Mean segment length : {duration_stats.get('mean_segment_duration_seconds', 0):.4f} s")
print()
print("Per-phoneme breakdown:")
print(f"  {'Phoneme':<10} {'Count':>5} {'Mean (ms)':>10} {'Total (ms)':>10}")
print(f"  {'-' * 10} {'-' * 5} {'-' * 10} {'-' * 10}")
for entry in duration_stats.get("phoneme_durations", []):
    phoneme = entry["phoneme"]
    count = entry["count"]
    mean_ms = entry["mean_duration_seconds"] * 1000
    total_ms = entry["total_duration_seconds"] * 1000
    print(f"  {phoneme:<10} {count:>5} {mean_ms:>10.1f} {total_ms:>10.1f}")

---

# Summary

In this lab, we used senselab to perform three complementary analyses
of the same speech recording:

| Analysis | Representation | senselab API |
|---|---|---|
| **ASR (Whisper)** | Text transcription | `transcribe_audios()` |
| **SPARC** | Articulatory trajectories (physical space) | `SparcFeatureExtractor.extract_sparc_features()` |
| **PPG** | Phoneme probabilities (symbolic space) | `extract_ppgs_from_audios()` |

Each approach captures different aspects of the speech signal:
- **ASR** gives you *what* was said (words)
- **SPARC** shows *how* it was produced (articulator movements)
- **PPG** reveals the *phonemic structure* over time

Together, these tools provide a rich, multi-level view of speech
production and perception -- all through senselab's reproducible,
device-aware pipelines.